# Sujet 5 · Le réseau des personnages des *Misérables* : centralités et communautés

**Cours « Intelligence artificielle »** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa

**Mini-projet, sujet 5** (analyse des réseaux sociaux) · **Durée :** environ 1 h 30

**Ce que vous allez faire**

- charger le réseau des personnages des *Misérables* de Victor Hugo (1862), fourni avec la bibliothèque NetworkX, et l'explorer personnage par personnage ;
- mesurer la place de chacun : centralité de degré (qui est le plus relié ?) et centralité d'intermédiarité (qui sert de pont ?), puis dessiner le réseau ;
- repérer des communautés de personnages et les interpréter en termes littéraires : intrigues, épisodes, rôle structurel des personnages.

**Mode d'emploi.** Exécutez les cellules de haut en bas : cliquez sur le bouton ▶ à gauche de chaque cellule, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu « Exécution → Tout exécuter » lance tout le carnet d'un coup.

Avant chaque cellule de code, une question vous demande de **prédire** le résultat : notez votre réponse, exécutez, puis comparez avec la rubrique « Ce que vous devez voir ». Python écrit les nombres décimaux avec un point : 0.57 se lit 0,57.

> **Les données.** Ce réseau n'est pas inventé : Donald Knuth l'a construit pour son livre *The Stanford GraphBase* (1993) en relevant, chapitre par chapitre, les personnages présents. Deux personnages sont reliés s'ils apparaissent dans un même chapitre ; le **poids** du lien est le nombre de chapitres qu'ils partagent. Le roman est dans le domaine public ; les noms sont écrits sans accents (`Thenardier` pour Thénardier).

## Étape 1 · Charger les outils

Cette cellule charge NetworkX (les graphes), pandas (les tableaux) et matplotlib (les dessins).

**Avant d'exécuter :** sous quel nom court utiliserons-nous NetworkX dans la suite du carnet ?

In [ ]:
#@title Charger les outils (bibliothèques)
import networkx as nx                 # graphes : nœuds et liens
import pandas as pd                   # tableaux de données
import matplotlib.pyplot as plt       # dessins

# Couleurs des communautés, toujours dans le même ordre
COULEURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
print("Outils chargés. Version de NetworkX :", nx.__version__)

**Ce que vous devez voir :** `Outils chargés. Version de NetworkX :` suivi d'un numéro de version (3.x ; le numéro exact dépend de votre installation). Nous écrirons `nx` pour NetworkX.

**Question :** un graphe est fait de « nœuds » et de « liens ». Dans un roman, que peuvent être les nœuds ? Que peuvent être les liens ? Donnez deux possibilités pour chacun.

## Étape 2 · Charger le réseau des *Misérables*

Cette cellule charge le réseau et affiche sa taille : le nombre de personnages (nœuds), le nombre de liens et la **densité**, c'est-à-dire la part des liens possibles qui existent vraiment (1 = tout le monde est relié à tout le monde).

**Avant d'exécuter :** combien de personnages Knuth a-t-il retenus, d'après vous : une trentaine, près de 80, ou plus de 200 ? La densité sera-t-elle proche de 0 ou de 1 ?

In [ ]:
#@title Charger le réseau et afficher sa taille
G = nx.les_miserables_graph()          # réseau fourni avec NetworkX
print("Personnages (nœuds) :", G.number_of_nodes())
print("Liens               :", G.number_of_edges())
print("Densité             :", round(nx.density(G), 3))
print("Quelques personnages :", ", ".join(sorted(G.nodes)[:12]), "…")

**Ce que vous devez voir :** 77 personnages, 254 liens et une densité de 0.087 : moins de 9 % des paires possibles de personnages se rencontrent dans un chapitre. Les premiers noms, par ordre alphabétique : Anzelma, Babet, Bahorel, Bamatabois, BaronessT, Blacheville, Bossuet, Boulatruelle, Brevet, Brujon, Champmathieu, Champtercier.

**Question :** un réseau de personnages peu dense est-il le signe d'un roman « mal construit » ? Pensez à la taille des *Misérables* et au nombre de lieux et d'époques qu'il traverse.

## Étape 3 · Explorer un personnage

Choisissez un personnage dans la liste (ou tapez un autre nom exactement comme dans les données). La cellule affiche le nombre de personnages auxquels il est relié, puis ses dix voisins les plus fréquents, avec le nombre de chapitres partagés.

**Avant d'exécuter :** avec qui Jean Valjean partage-t-il le plus de chapitres ? Javert, Cosette ou Marius ?

In [ ]:
#@title À MODIFIER : choisissez un personnage, puis exécutez (ses voisins)
personnage = "Valjean" #@param ["Valjean", "Javert", "Cosette", "Marius", "Fantine", "Gavroche", "Myriel", "Thenardier", "Eponine", "Enjolras"] {allow-input: true}
voisins = G[personnage]                # les personnages reliés à celui-ci
print(personnage, "est relié à", len(voisins), "personnages.")
tableau = pd.DataFrame([(v, voisins[v]["weight"]) for v in voisins],
                       columns=["voisin", "chapitres partagés"])
print(tableau.sort_values("chapitres partagés", ascending=False, kind="stable").head(10).to_string(index=False))

**Ce que vous devez voir :** `Valjean est relié à 36 personnages.` Ses voisins les plus fréquents : Cosette (31 chapitres), Marius (19), Javert (17), Thenardier (12), Fantine (9), Fauchelevent (8), MmeThenardier (7), Myriel (5), Enjolras (4), MmeMagloire (3). C'est Cosette, la fille adoptive, et non Javert, le poursuivant, qui accompagne le plus souvent Valjean.

**Question :** choisissez Javert, puis Gavroche. Lequel a le plus de voisins ? Les voisins de Gavroche appartiennent-ils au même « monde » que ceux de Javert ?

## Étape 4 · La centralité de degré : qui est le plus relié ?

Le **degré** d'un personnage est le nombre de personnages auxquels il est relié. La **centralité de degré** le divise par le nombre d'autres personnages (76) : c'est la part du réseau que le personnage rencontre directement.

**Avant d'exécuter :** qui sera deuxième derrière Valjean ? Un personnage principal (Marius, Cosette, Javert) ou un personnage que l'on croise partout ?

In [ ]:
#@title Calculer le degré et la centralité de degré de chaque personnage
liens = dict(G.degree())                        # nombre de personnages reliés
centralites = pd.DataFrame({"liens": liens, "degre": nx.degree_centrality(G)}).round(3)
top = centralites.sort_values("liens", ascending=False, kind="stable").head(10)
print(top)
a_dessiner = top.sort_values("liens")           # du plus petit au plus grand, pour le dessin
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(a_dessiner.index, a_dessiner["liens"], color=COULEURS[0])
ax.set_xlabel("Nombre de personnages reliés (degré)")
ax.set_title("Les 10 personnages les plus reliés")
plt.show()

**Ce que vous devez voir :** Valjean (36 liens, centralité 0.474 : il rencontre près de la moitié des personnages), puis Gavroche (22), Marius (19), Javert (17), Thenardier (16), Fantine et Enjolras (15), Courfeyrac et Bossuet (13), Bahorel (12, à égalité avec Joly, qui reste hors du tableau). Cosette n'est pas dans les dix premiers.

**Question :** Gavroche passe devant Marius, Javert et Cosette. Qu'est-ce qui, dans son rôle de « gamin de Paris », explique qu'il rencontre autant de personnages ?

## Étape 5 · La centralité d'intermédiarité : qui sert de pont ?

Pour aller d'un personnage à un autre dans le réseau, on suit le plus court chemin de liens. La **centralité d'intermédiarité** (*betweenness*) mesure la part de ces plus courts chemins qui passent par un personnage : un personnage à forte intermédiarité relie des groupes qui, sans lui, ne se toucheraient pas.

**Avant d'exécuter :** un personnage qui n'apparaît que dans le premier livre du roman peut-il avoir une forte intermédiarité ?

In [ ]:
#@title Calculer la centralité d'intermédiarité
centralites["intermediarite"] = pd.Series(nx.betweenness_centrality(G)).round(3)
centralites.sort_values("intermediarite", ascending=False).head(10)

**Ce que vous devez voir :** Valjean écrase tout (0.570 : plus de la moitié des plus courts chemins passent par lui). Viennent ensuite **Myriel** (0.177), Gavroche (0.165), Marius (0.132), Fantine (0.130), Thenardier (0.075), Javert (0.054), MlleGillenormand (0.048), Enjolras (0.043) et Tholomyes (0.041). Myriel, l'évêque de Digne, n'a que 10 liens mais il est le deuxième « pont » du roman.

**Question :** Javert a 17 liens mais une intermédiarité faible (0.054). Que peut-on en conclure sur sa place dans la structure du roman, par rapport à celle de Myriel ?

Ce graphique place chaque personnage selon ses deux centralités : le degré en abscisse, l'intermédiarité en ordonnée. Les noms des huit plus grands « ponts » sont écrits.

**Avant d'exécuter :** où sera Myriel sur le graphique : en haut à droite, en haut à gauche, ou en bas ?

In [ ]:
#@title Comparer les deux centralités sur un graphique
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(centralites["degre"], centralites["intermediarite"], s=40, color=COULEURS[0], alpha=0.7)
for nom in centralites.sort_values("intermediarite").tail(8).index:    # les 8 plus grands « ponts »
    ax.annotate(nom, (centralites.loc[nom, "degre"], centralites.loc[nom, "intermediarite"]),
                xytext=(5, 3), textcoords="offset points")
ax.set_xlabel("Centralité de degré (part des personnages rencontrés)")
ax.set_ylabel("Centralité d'intermédiarité (rôle de pont)")
ax.margins(x=0.12)                     # de la place pour les noms au bord
ax.set_title("Être très relié ou servir de pont ?")
plt.show()

**Ce que vous devez voir :** un nuage de 77 points serrés en bas à gauche (la plupart des personnages sont peu reliés et ne servent de pont à personne), Valjean seul tout en haut à droite, et Myriel **en haut à gauche** : peu de liens, beaucoup de passages. Gavroche, Marius et Fantine forment un groupe intermédiaire.

**Question :** en termes littéraires, que signifie la position de Myriel ? Pensez au premier livre du roman (« Un juste ») et à la scène des chandeliers.

## Étape 6 · Dessiner le réseau

NetworkX place les personnages avec la disposition de Kamada et Kawai : la distance entre deux points sur le dessin essaie de refléter le nombre de liens qu'il faut parcourir pour aller de l'un à l'autre. Cette méthode ne tire rien au hasard : le dessin est le même à chaque exécution. La taille d'un point dépend de son degré, l'épaisseur d'un trait du nombre de chapitres partagés. Seuls les personnages qui ont au moins 10 liens sont nommés.

**Avant d'exécuter :** Valjean sera-t-il au centre du dessin ou sur un bord ? Verra-t-on des groupes séparés ?

In [ ]:
#@title Dessiner le réseau (taille = nombre de liens, épaisseur = chapitres partagés)
position = nx.kamada_kawai_layout(G, weight=None)       # distance ≈ nombre de liens à parcourir
etiquettes = {p: (x, y + 0.06) for p, (x, y) in position.items()}   # noms un peu au-dessus des points
tailles = [25 * liens[p] for p in G.nodes]
epaisseurs = [0.3 + G[u][v]["weight"] / 6 for u, v in G.edges]
principaux = {p: p for p in G.nodes if liens[p] >= 10}  # noms affichés
fond_blanc = dict(facecolor="white", edgecolor="none", alpha=0.7)    # fond des noms
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(G, position, width=epaisseurs, alpha=0.4, edge_color="gray", ax=ax)
nx.draw_networkx_nodes(G, position, node_size=tailles, node_color=COULEURS[0], ax=ax)
nx.draw_networkx_labels(G, etiquettes, labels=principaux, font_size=9, bbox=fond_blanc, ax=ax)
ax.set_title("Les Misérables : qui apparaît avec qui ? (données de Knuth, 1993)")
ax.axis("off")
plt.show()

**Ce que vous devez voir :** un réseau en étoile autour de Valjean, le plus gros point, au centre. Autour de lui, des grappes : en bas, un paquet serré de personnages très reliés entre eux (Enjolras, Courfeyrac, Bossuet, Joly, Combeferre… les Amis de l'ABC, avec Gavroche) ; en haut à gauche, un groupe autour de Fantine ; en haut à droite, Myriel entouré de petits points qui ne sont reliés qu'à lui ; au centre, Javert, Cosette, les Thénardier et Marius, tout près de Valjean. Le trait le plus épais relie Valjean à Cosette. (Selon votre version de NetworkX, le dessin peut être tourné ou retourné, mais les groupes restent les mêmes.)

**Question :** le trait le plus épais relie Valjean et Cosette (31 chapitres). Quel autre trait épais repérez-vous, et quelle relation du roman représente-t-il ?

## Étape 7 · Les communautés

Une **communauté** est un groupe de personnages très reliés entre eux et peu reliés au reste. La fonction `greedy_modularity_communities` les cherche en fusionnant pas à pas les groupes, tant que la **modularité** (une note qui mesure si les liens restent plutôt à l'intérieur des groupes) augmente. Vous pouvez choisir de tenir compte des poids (le nombre de chapitres partagés) ou de compter tous les liens de la même façon.

**Avant d'exécuter :** combien de communautés attendez-vous ? Valjean et Javert seront-ils dans la même ?

In [ ]:
#@title À MODIFIER : tenir compte des poids ou non, puis exécutez (communautés)
avec_poids = True #@param {type:"boolean"}
poids = "weight" if avec_poids else None     # None : tous les liens comptent pareil
communautes = nx.community.greedy_modularity_communities(G, weight=poids)
print(len(communautes), "communautés, modularité =",
      round(nx.community.modularity(G, communautes, weight=poids), 3))
for numero, membres in enumerate(communautes):
    print(f"\nCommunauté {numero} ({len(membres)} personnages) :")
    print("   ", ", ".join(sorted(membres)))

**Ce que vous devez voir :** `5 communautés, modularité = 0.547`, puis la liste des membres :

- **Communauté 0** (33 personnages) : Valjean, Cosette, Marius, Javert, les Thénardier et leurs filles (Eponine, Anzelma), la famille Gillenormand, Fauchelevent, la bande de Patron-Minette (Babet, Claquesous, Gueulemer, Montparnasse, Brujon)… ;
- **Communauté 1** (17) : Enjolras, Combeferre, Courfeyrac, Bahorel, Bossuet, Joly, Grantaire, Feuilly, Prouvaire, Gavroche, Mabeuf, MmeHucheloup… ;
- **Communauté 2** (11) : Fantine, Tholomyes et ses amis (Blacheville, Dahlia, Fameuil, Favourite, Listolier, Zephine), Marguerite, Perpetue, Simplice ;
- **Communauté 3** (10) : Myriel, MlleBaptistine, MmeMagloire, Napoleon, Cravatte, Champtercier… ;
- **Communauté 4** (6) : Champmathieu, Brevet, Chenildieu, Cochepaille, Bamatabois, Judge.

**Question :** dans quelle communauté se trouve Marius ? Et Enjolras ? Qu'est-ce que cette séparation dit du déchirement de Marius entre ses amis révolutionnaires et son amour pour Cosette ?

Cette cellule redessine le réseau avec la **même disposition** qu'à l'étape 6, mais chaque personnage prend la couleur de sa communauté.

**Avant d'exécuter :** les communautés formeront-elles des zones bien séparées sur le dessin, ou seront-elles mélangées ?

In [ ]:
#@title Dessiner le réseau coloré par communauté
couleur_de = {}
for numero, membres in enumerate(communautes):
    for p in membres:
        couleur_de[p] = COULEURS[numero]
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(G, position, width=epaisseurs, alpha=0.3, edge_color="gray", ax=ax)
nx.draw_networkx_nodes(G, position, node_size=tailles, node_color=[couleur_de[p] for p in G.nodes], ax=ax)
nx.draw_networkx_labels(G, etiquettes, labels=principaux, font_size=9, bbox=fond_blanc, ax=ax)
for numero in range(len(communautes)):                  # légende : une couleur par communauté
    ax.scatter([], [], color=COULEURS[numero], label=f"Communauté {numero}")
ax.legend(loc="lower left")
ax.set_title(f"{len(communautes)} communautés dans le réseau des Misérables")
ax.axis("off")
plt.show()

**Ce que vous devez voir :** le même dessin, en cinq couleurs qui occupent des zones distinctes : bleu au centre (Valjean et l'intrigue principale), orange en bas (l'ABC et Gavroche), vert en haut à gauche (Fantine et ses amis de jeunesse), jaune en haut à droite (Myriel et Digne), rose en haut (le procès Champmathieu). La communauté jaune ne touche le reste du réseau que par Valjean.

**Question :** repérez sur le dessin une communauté qui ne touche le reste du réseau que par un ou deux personnages. Lesquels ?

### Lire les communautés en termes littéraires

Les chiffres ne disent rien par eux-mêmes ; confrontés au roman, ils éclairent sa construction.

- **Un roman en étoile.** Valjean a 36 liens et une intermédiarité de 0,57 : plus de la moitié des plus courts chemins passent par lui. *Les Misérables* juxtapose des mondes (Digne, Montreuil-sur-Mer, le Paris des Thénardier, celui des Gillenormand, la barricade de 1832) que seul le parcours de Valjean relie : sans lui, le réseau se défait (étape 8).
- **Les communautés retrouvent les épisodes.** La communauté de Myriel correspond au premier livre, « Un juste », presque un récit à part ; celle de Fantine rassemble la première partie (Tholomyès et ses amis dans « En l'année 1817 », puis Montreuil-sur-Mer et l'infirmerie de sœur Simplice) ; le procès Champmathieu forme un îlot, le temps d'« Une tempête sous un crâne » ; les Amis de l'ABC, Gavroche et Mabeuf forment le monde de l'insurrection. L'algorithme ne connaît ni les livres ni les parties du roman : il retrouve la composition de Hugo à partir des seules coprésences.
- **Des ponts qui font sens.** Myriel, peu relié mais deuxième « pont », est l'origine morale du roman : c'est par lui que Valjean entre dans l'histoire (les chandeliers). Gavroche circule entre les mondes : fils des Thénardier, compagnon des étudiants, protecteur des deux petits enfants (Child1 et Child2) qu'il abrite dans l'éléphant de la Bastille, et qui sont ses frères sans qu'il le sache. Fantine relie sa jeunesse parisienne au monde de Valjean.
- **Marius, personnage partagé.** Rangé avec Cosette et les Gillenormand plutôt qu'avec ses amis de l'ABC, Marius garde une forte intermédiarité (0,132) : le réseau dessine son déchirement entre l'amour et la barricade.
- **Centralité structurelle et importance pour le sens.** Javert, pilier dramatique du roman, n'est qu'un pont modeste (0,054) : il suit Valjean partout, mais ne relie pas de mondes que Valjean ne relie déjà. Cosette n'est même pas parmi les dix personnages les plus reliés. Et la règle de Knuth a ses angles morts : être dans le même chapitre n'est pas se rencontrer, et les grandes digressions (Waterloo, le couvent, les égouts de Paris), où les personnages se font rares, disparaissent presque du réseau.

## Étape 8 · Retirer un personnage : l'expérience de Moretti

Dans « Network Theory, Plot Analysis » (2011), Franco Moretti a représenté *Hamlet* comme un réseau, puis observé ce que devient la pièce quand on retire un personnage. Faisons de même : la cellule retire le personnage choisi et compte les **morceaux** du réseau qui ne sont plus reliés entre eux.

**Avant d'exécuter :** sans Valjean, le réseau restera-t-il d'un seul tenant ? Si non, quels personnages se retrouveront isolés ?

In [ ]:
#@title À MODIFIER : retirez un personnage, puis exécutez
retirer = "Valjean" #@param ["Valjean", "Javert", "Cosette", "Marius", "Fantine", "Gavroche", "Myriel", "Thenardier", "Eponine", "Enjolras"] {allow-input: true}
H = G.copy()                           # une copie : le réseau G reste intact
H.remove_node(retirer)
morceaux = sorted(nx.connected_components(H), key=len, reverse=True)
print(f"Sans {retirer}, le réseau se coupe en {len(morceaux)} morceau(x).")
print("Le plus grand morceau garde", len(morceaux[0]), "personnages sur", H.number_of_nodes())
for morceau in morceaux[1:]:
    print("   détaché :", ", ".join(sorted(morceau)))

**Ce que vous devez voir :** sans Valjean, le réseau se coupe en **7 morceaux**. Le plus grand garde 61 personnages sur 76. Se détachent tout le monde de Digne (Myriel, MlleBaptistine, MmeMagloire, Napoleon, Champtercier, Cravatte, Count, CountessDeLo, Geborand, OldMan) et cinq personnages que seul Valjean rencontre : Gervais (le petit Savoyard dont il vole la pièce), Isabeau, Labarre (l'aubergiste de Digne qui le chasse), MmeDeR et Scaufflaire (qui lui loue le cabriolet pour Arras).

**Question :** retirez maintenant Myriel, puis Gavroche, puis Marius. Lequel de ces retraits ne coupe **rien** ? Qu'est-ce que cela dit de la différence entre un personnage important pour l'intrigue et un personnage indispensable à la structure ?

## À vous

Modifiez seulement les champs, puis relancez les cellules à partir de celle que vous avez changée (dans Colab : « Exécution → Exécuter les cellules suivantes »).

1. **Étape 3 :** comparez les voisins de Fantine et de Cosette, la mère et la fille. Partagent-elles des voisins ? Qu'est-ce que cela dit de la construction du roman en époques successives ?
2. **Étape 7 :** décochez `avec_poids` et relancez les étapes 7 et suivantes. Combien de communautés obtenez-vous ? Où vont Myriel et Fantine ? Quelle nouvelle communauté apparaît autour de Marius et Cosette ?
3. **Étape 8 :** tapez `Fantine`, puis `Thenardier` dans le champ (ou choisissez-les dans la liste). Leur retrait coupe-t-il le réseau ? Comparez avec leur intermédiarité (étape 5).

## Pour le mini-projet : votre propre réseau et Gephi

Le sujet 5 propose par exemple d'étudier le réseau des personnages de *Nedjma* (Kateb Yacine, 1956). Le roman est protégé par le droit d'auteur : partagez vos tableaux de liens, jamais le texte numérisé. Pour construire votre réseau :

- écrivez d'abord votre **règle** : qu'est-ce qu'un personnage, qu'est-ce qu'un lien (apparaître dans la même section, se parler, s'écrire) ?
- dans un tableur, faites un tableau de liens avec **trois colonnes** nommées `source`, `cible` et `poids` (une ligne par paire de personnages, le poids = nombre de sections communes), puis enregistrez-le en CSV ;
- cochez la case ci-dessous et exécutez : dans Colab, une fenêtre vous demande le fichier ; dans Jupyter, placez-le à côté du carnet sous le nom `mes_liens.csv` ;
- relancez ensuite les étapes 3 à 8, en tapant dans les champs des étapes 3 et 8 le nom d'un de vos personnages.

**Avant d'exécuter :** si vous laissez la case décochée, que va faire la cellule ?

In [ ]:
#@title À MODIFIER : charger votre propre tableau de liens (fichier CSV)
charger_mon_fichier = False #@param {type:"boolean"}
if charger_mon_fichier:
    try:
        from google.colab import files       # n'existe que dans Colab
        envoi = files.upload()               # fenêtre pour choisir le fichier
        nom_fichier = list(envoi.keys())[0]
    except ImportError:                      # Jupyter : fichier placé à côté du carnet
        nom_fichier = "mes_liens.csv"
    tableau_liens = pd.read_csv(nom_fichier).rename(columns={"poids": "weight"})
    G = nx.from_pandas_edgelist(tableau_liens, "source", "cible", edge_attr="weight")
    print("Votre réseau :", G.number_of_nodes(), "personnages,", G.number_of_edges(), "liens")
else:
    print("Case non cochée : le carnet garde le réseau des Misérables.")

**Ce que vous devez voir :** avec la case décochée, `Case non cochée : le carnet garde le réseau des Misérables.` Avec votre fichier, le nombre de personnages et de liens de votre réseau.

Le mini-projet demande aussi une visualisation avec **Gephi**, un logiciel libre qui met en forme les réseaux. La cellule suivante enregistre le réseau (avec le numéro de communauté de chaque personnage) au format GEXF, que Gephi sait ouvrir.

**Avant d'exécuter :** où le fichier sera-t-il enregistré dans Colab ? Et dans Jupyter ?

In [ ]:
#@title À MODIFIER : exporter le réseau pour Gephi (fichier GEXF)
exporter = False #@param {type:"boolean"}
if exporter:
    communautes = nx.community.greedy_modularity_communities(G, weight=poids)   # réseau actuel
    for numero, membres in enumerate(communautes):
        for p in membres:
            G.nodes[p]["communaute"] = numero        # Gephi pourra colorer par communauté
    nx.write_gexf(G, "reseau.gexf")
    try:
        from google.colab import files
        files.download("reseau.gexf")              # Colab : téléchargement sur votre ordinateur
    except ImportError:
        print("Fichier reseau.gexf enregistré à côté du carnet.")
else:
    print("Case non cochée : aucun fichier n'est créé.")

**Ce que vous devez voir :** avec la case décochée, `Case non cochée : aucun fichier n'est créé.` Cochée, le fichier `reseau.gexf` est téléchargé (Colab) ou enregistré à côté du carnet (Jupyter). Dans Gephi : « Fichier → Ouvrir », puis colorez les nœuds selon l'attribut `communaute`.

**Question :** NetworkX calcule, Gephi met en forme. Quel avantage voyez-vous à garder les calculs dans un carnet que d'autres peuvent relancer ?

## À rendre

Répondez par écrit, en français, en quelques lignes pour chaque question.

1. **Centralités.** Comparez Valjean, Myriel, Javert et Gavroche à l'aide du degré et de l'intermédiarité. Pour chacun, dites en une phrase ce que ces chiffres révèlent de son rôle dans la structure du roman.
2. **Communautés.** Choisissez deux communautés trouvées à l'étape 7. Rattachez chacune à un épisode, un lieu ou une intrigue des *Misérables*, et signalez un personnage dont la place vous surprend.
3. **Limites.** Knuth relie deux personnages présents dans le même chapitre. Donnez deux limites de cette règle (par exemple : se trouver dans le même chapitre sans se rencontrer, chapitres de longueurs très différentes, digressions presque sans personnages comme Waterloo) et proposez une autre règle de lien pour votre propre corpus.

## Ce qu'il faut retenir

- Un **graphe** représente des relations : des **nœuds** (personnages) reliés par des **liens** (ici, apparaître dans le même chapitre), avec un **poids**. Tout dépend de la règle qui définit un lien.
- La **centralité de degré** mesure combien de personnages on rencontre ; la **centralité d'intermédiarité** mesure le rôle de **pont** entre des groupes. Un personnage peu relié peut être un pont essentiel (Myriel).
- Les **communautés** font apparaître des groupes de personnages qui correspondent souvent à des épisodes ou à des intrigues ; les chiffres ne remplacent pas la lecture : il faut les confronter au texte.

**Leçon du site :** [Mini-projet, sujet 5 : analyse des réseaux sociaux](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-5)